# Classical → Quantum → Classical: Dialect Round-Trip

This notebook shows how to **mix dialects and ops** in a single traced module to move data from classical preparation through a quantum circuit and back to classical post-processing.

### Pipeline

```
Classical prep (ops.arith, ops.linalg, gpu dialect)
 │
 ▼
Encode into quantum circuit (quantum dialect: Ry, Rz rotations)
 │
 ▼
Entangle + process (quantum.cnot, quantum.cz, quantum.h)
 │
 ▼
Measure (quantum.measure → classical bits)
 │
 ▼
Classical post-processing (ops.arith, ops.trans, gpu dialect)
```

### What you'll learn

1. How `@tracing.to_module` captures ops from **multiple dialects** into one IR module
2. How to encode classical parameters into quantum rotation angles
3. How `quantum.measure()` returns classical booleans that feed into classical ops
4. How uniqx routes each node to the right hardware (CPU, GPU, qsim)
5. How `preflight()` reveals the hardware assignment for every node

In [ ]:
# Copyright (c) 2024-2026 ORIQX AG/LTD/SAS. All rights reserved.

import os
import math
import numpy as np

# Core SDK
from uniqx import ops, tracing, types, login
from uniqx import connect, submit, get, preflight, parse_buffer_view

# Dialects
from uniqx.dialects import quantum, gpu, tpu

# For the prod gateway: export UNIQX_GATEWAY=api.oriqx.com:443 and UNIQX_API_KEY=...
endpoint = os.environ.get("UNIQX_GATEWAY", "api.oriqx.com:443")
login(os.environ["UNIQX_API_KEY"], gateway=endpoint)
client = connect(endpoint)
print("Connected to", endpoint)

---
## Part 1: Classical Data Preparation

We start with a classical vector and compute rotation angles for a quantum circuit. This uses **core arithmetic ops** and the **GPU dialect** for the heavy linear algebra.

The idea: given a 2D feature vector `[x, y]`, encode it into qubit rotation angles via `theta = atan2(y, x)` and `phi = sqrt(x² + y²)`.

In [ ]:
@tracing.to_module(name="classical_prep")
def classical_prep(features, weight_matrix):
    """Classical data preparation: transform features with a weight matrix,
    then compute rotation angles for quantum encoding.

    Uses: ops.add/mul, ops.atan2, ops.sqrt, gpu.gemv
    """
    # GPU-accelerated matrix-vector multiply: project features
    projected = gpu.gemv(weight_matrix, features)

    # Element-wise square of the projected features
    sq_proj = ops.mul(projected, projected)

    # Rotation angle from feature vs projected components
    theta = ops.atan2(features, projected)

    # Radius: sqrt(projected² + features²)
    phi = ops.sqrt(ops.add(sq_proj, ops.mul(features, features)))

    return theta, phi


# Input data
features = [0.6, 0.8]            # 2D feature vector
W = [[0.5, -0.3], [-0.3, 0.7]]   # 2x2 weight matrix

mod_prep = classical_prep(features, W)
print(f"Classical prep module: {len(mod_prep.to_text())} chars IR")
print(f"\nIR preview:\n{mod_prep.to_text()[:400]}...")

In [ ]:
pf = preflight(mod_prep, client=client)
print("Preflight for classical prep:")
print(pf.summary())
print(f"\nNeeds GPU: {pf.needs_gpu}")

---
## Part 2: Quantum Circuit with Gate-Level Dialect

Now we build a parameterised quantum circuit using the **quantum dialect**. This traces individual gate operations (Ry, Rz, CNOT, H) into the IR.

The circuit:
1. **Encode** classical angles into qubit rotations (Ry, Rz)
2. **Entangle** with CNOT + CZ layers
3. **Measure** both qubits back to classical bits

In [ ]:
@tracing.to_module(name="quantum_circuit")
def quantum_circuit(q0, q1):
    """Parameterised 2-qubit circuit.

    Uses: quantum.ry, quantum.rz, quantum.h, quantum.cnot, quantum.cz, quantum.measure
    """
    # --- Encoding layer: map classical angles to qubit states ---
    q0 = quantum.ry(0.6, q0)   # angle from classical prep
    q1 = quantum.ry(0.8, q1)
    q0 = quantum.rz(0.3, q0)
    q1 = quantum.rz(0.5, q1)

    # --- Entangling layer ---
    q1 = quantum.cnot(q0, q1)

    # --- Processing layer ---
    q0 = quantum.h(q0)
    q1 = quantum.rz(0.2, q1)

    # --- Second entangling layer ---
    q0 = quantum.cz(q0, q1)

    # --- Measurement: quantum -> classical ---
    m0 = quantum.measure(q0)
    m1 = quantum.measure(q1)

    return m0, m1


mod_qc = quantum_circuit(types.qubit(), types.qubit())
print(f"Quantum circuit module: {len(mod_qc.to_text())} chars IR")
print(f"\nFull IR:\n{mod_qc.to_text()}")

In [ ]:
pf_qc = preflight(mod_qc, client=client)
print("Preflight for quantum circuit:")
print(pf_qc.summary())

# Inspect node assignments
for i, opt in enumerate(pf_qc):
    assignments = opt.get("node_assignments", {})
    hw_set = set(assignments.values())
    print(f"  [{i}] {opt.get('label', 'opt-' + str(i))}: hardware targets = {hw_set}")

---
## Part 3: Classical Post-Processing

After measurement, we take the classical bits and compute derived quantities. This uses **core ops** and the **GPU dialect** for the final computation.

In [ ]:
@tracing.to_module(name="classical_postprocess")
def classical_postprocess(measurement_a, measurement_b, scale_matrix):
    """Post-process quantum measurement results with classical computation.

    Uses: ops.add, ops.tanh, ops.logistic, gpu.gemv
    """
    # Combine measurements into a feature vector
    combined = ops.add(measurement_a, measurement_b)

    # Apply nonlinear activation (classical neural-net style)
    activated = ops.tanh(combined)

    # Scale with GPU matrix-vector multiply
    scaled = gpu.gemv(scale_matrix, activated)

    # Final sigmoid output
    output = ops.logistic(scaled)

    return output


# Dummy inputs for tracing (real values come from quantum measurement)
m_a = [1.0, 0.0]
m_b = [0.0, 1.0]
S = [[0.8, 0.2], [0.2, 0.8]]

mod_post = classical_postprocess(m_a, m_b, S)
print(f"Post-processing module: {len(mod_post.to_text())} chars IR")

---
## Part 3b: TPU Dialect — Batch Parameter Evaluation

TPUs excel at large-batch matrix operations. Here we use `tpu.matmul` to evaluate multiple parameter configurations in parallel — the kind of workload where TPU MXUs shine. This could feed rotation angles into a quantum circuit sweep.

In [ ]:
@tracing.to_module(name="tpu_batch_params")
def tpu_batch_params(param_grid, transform_matrix):
    """Batch-evaluate parameter configurations on TPU.

    param_grid:       [n_configs, dim] — each row is a parameter vector
    transform_matrix: [dim, dim]       — projection matrix

    Uses: tpu.matmul for large batch matmul, tpu.reduce_sum for aggregation
    """
    # TPU batch matmul: project all parameter vectors at once
    projected = tpu.matmul(param_grid, transform_matrix)

    # TPU reduction: sum across the batch dimension
    total = tpu.reduce_sum(projected, axis=0)

    return projected, total


# Create a grid of trial parameter vectors
n_configs = 16
dim = 4
rng = np.random.default_rng(42)
params = rng.normal(size=(n_configs, dim))
params = params / np.linalg.norm(params, axis=1, keepdims=True)

T = np.eye(dim) * 0.5 + rng.normal(size=(dim, dim)) * 0.1  # near-identity transform

mod_tpu = tpu_batch_params(params.tolist(), T.tolist())
print(f"TPU batch module: {len(mod_tpu.to_text())} chars IR")

pf_tpu = preflight(mod_tpu, client=client)
print("\nPreflight for TPU batch evaluation:")
print(pf_tpu.summary())

---
## Part 4: Full Round-Trip in One Module

Now we combine **all three stages** into a single traced module. This is the key pattern: one `@tracing.to_module` function that mixes `ops.arith`, `ops.trans`, `gpu.*`, and `quantum.*` seamlessly.

uniqx analyses the full DAG, partitions nodes by hardware, and executes each partition on the optimal target.

In [ ]:
@tracing.to_module(name="full_roundtrip")
def full_roundtrip(features, weights, q0, q1):
    """Complete classical -> quantum -> classical pipeline.

    Stage 1 (classical): Transform features with GPU-accelerated matmul
    Stage 2 (quantum):   Encode into circuit, entangle, measure
    Stage 3 (classical): Post-process measurements with activation functions

    Dialects used: ops.arith, ops.trans, gpu, quantum
    """
    # ===== STAGE 1: Classical preparation =====
    projected = gpu.gemv(weights, features)

    norm_sq = ops.add(
        ops.mul(projected, projected),
        ops.mul(features, features),
    )
    scale = ops.sqrt(norm_sq)

    # ===== STAGE 2: Quantum circuit =====
    q0 = quantum.ry(0.6, q0)
    q1 = quantum.ry(0.8, q1)
    q0 = quantum.rz(0.3, q0)
    q1 = quantum.rz(0.5, q1)

    q1 = quantum.cnot(q0, q1)
    q0 = quantum.h(q0)
    q1 = quantum.rz(0.2, q1)
    q0 = quantum.cz(q0, q1)

    m0 = quantum.measure(q0)
    m1 = quantum.measure(q1)

    # ===== STAGE 3: Classical post-processing =====
    # The scale factor from Stage 1 flows through to the output alongside
    # the classical measurement bits.
    return m0, m1, scale


mod_full = full_roundtrip(
    features,         # classical 2D vector
    W,                # weight matrix
    types.qubit(),    # quantum register
    types.qubit(),
)

print(f"Full round-trip module: {len(mod_full.to_text())} chars IR")
print(f"\nIR preview:\n{mod_full.to_text()[:600]}...")

---
## Part 5: All Dialects + Primitives in One Module

Here we combine **every dialect** and **primitives** in a single traced module:
- **Primitives** (`expv`, `expect`) for Hamiltonian simulation
- **GPU dialect** (`gemm`, `syevd`) for accelerated linear algebra
- **TPU dialect** (`matmul`) for batch parameter evaluation
- **Quantum dialect** (`h`, `cnot`, `measure`) for circuit execution
- **Core ops** (`transpose`) for glue

uniqx analyses the full DAG and assigns each node to the optimal hardware.

In [ ]:
# Pauli matrices for Hamiltonian construction
X = [[0, 1], [1, 0]]
Z = [[1, 0], [0, -1]]
I2 = [[1, 0], [0, 1]]


@tracing.to_module(name="all_dialects_pipeline")
def all_dialects_pipeline(H, psi, t, basis_transform, param_batch, q0, q1):
    """Pipeline using ALL dialects + primitives in one module.

    1. [primitive] Time-evolve a state under a Hamiltonian
    2. [primitive] Measure energy expectation value
    3. [gpu] Rotate Hamiltonian into a new basis + eigendecompose
    4. [tpu] Batch matmul for multi-config parameter evaluation
    5. [quantum] Prepare a Bell pair and measure
    """
    # --- Primitives: time evolution ---
    evolved = ops.expv(H, psi, t, hermitian=True)
    energy = ops.expect(H, evolved)

    # --- GPU dialect: basis transform + eigendecompose ---
    bt = ops.transpose(basis_transform, permutation=[1, 0])
    tmp = gpu.gemm(H, basis_transform)
    h_rotated = gpu.gemm(bt, tmp)
    spectrum = gpu.syevd(h_rotated)

    # --- TPU dialect: batch parameter evaluation ---
    batch_energies = tpu.matmul(param_batch, H)

    # --- Quantum dialect: Bell pair ---
    q0 = quantum.h(q0)
    q1 = quantum.cnot(q0, q1)
    m0 = quantum.measure(q0)
    m1 = quantum.measure(q1)

    return energy, spectrum, batch_energies, m0, m1


# Build a simple 2-qubit Hamiltonian
H_np = np.array(np.kron(Z, Z), dtype=float) + 0.5 * np.array(np.kron(X, I2), dtype=float)
psi0 = [1.0, 0.0, 0.0, 0.0]  # |00> state

# Basis rotation
theta = math.pi / 6
c, s = math.cos(theta), math.sin(theta)
B = [[c, -s, 0, 0], [s, c, 0, 0], [0, 0, c, -s], [0, 0, s, c]]

# Batch of trial state vectors for TPU
trial_states = rng.normal(size=(8, 4)).tolist()

mod_all = all_dialects_pipeline(
    H_np.tolist(), psi0, 0.5, B, trial_states,
    types.qubit(), types.qubit(),
)
print(f"All-dialects module: {len(mod_all.to_text())} chars IR")

In [ ]:
pf_all = preflight(mod_all, client=client)
print("Preflight for all-dialects pipeline:")
print(pf_all.summary())

print("\nDetailed node assignments:")
for i, opt in enumerate(pf_all):
    label = opt.get("label", f"opt-{i}")
    assignments = opt.get("node_assignments", {})
    algorithms = opt.get("node_algorithms", {})
    rec = " <-- recommended" if opt.get("recommended") else ""

    hw_counts = {}
    for hw in assignments.values():
        hw_counts[hw] = hw_counts.get(hw, 0) + 1

    print(f"\n  [{i}] {label}{rec}")
    print(f"       Hardware counts: {hw_counts}")
    print(f"       Time: {opt.get('total_time', 0):.4f}s")
    print(f"       Cost: ${opt.get('total_cost_usd', 0):.6f}")
    if algorithms:
        print(f"       Algorithms: {dict(list(algorithms.items())[:5])}...")

In [ ]:
# Execute the all-dialects pipeline. Some primitive/dialect combinations
# may not yet be supported on every backend; guard the execution path.
labels = [
    "energy (primitive)",
    "spectrum (gpu)",
    "batch_energies (tpu)",
    "m0 - qubit 0 (quantum)",
    "m1 - qubit 1 (quantum)",
]

try:
    jid = submit(mod_all, client=client)
    result = get(jid, client=client)

    payload = result.get("result_payload") or result.get("payload", b"")
    if isinstance(payload, bytes):
        payload = payload.decode("utf-8")

    output_lines = [ln for ln in payload.strip().split("\n") if "=" in ln]
    if not output_lines:
        print("Known limitation: all-dialects pipeline not yet runnable on this backend; preflight succeeded but no payload was returned.")
    else:
        print("All-dialects pipeline results:")
        for i, line in enumerate(output_lines):
            label = labels[i] if i < len(labels) else f"output_{i}"
            parsed = parse_buffer_view(line)
            if parsed is None:
                print(f"  {label}: <unavailable>")
                continue
            _, _, vals = parsed
            if not vals:
                print(f"  {label}: <empty>")
            elif len(vals) <= 8:
                print(f"  {label}: {vals}")
            else:
                print(f"  {label}: [{vals[0]:.4f}, ..., {vals[-1]:.4f}] ({len(vals)} values)")
except Exception as exc:
    print(f"Known limitation: all-dialects pipeline not yet runnable on this backend ({exc.__class__.__name__}).")

---
## Summary

### Dialect + Ops Mixing Rules

| Source | Import | Modality | Hardware |
|:-------|:-------|:---------|:---------|
| Core ops | `from uniqx import ops` | auto | CPU (default) |
| Arithmetic | `ops.add`, `ops.mul`, ... | auto | CPU or GPU |
| Transcendental | `ops.sin`, `ops.exp`, `ops.tanh`, ... | auto | CPU or GPU |
| Linear algebra | `ops.matmul`, `ops.kron`, ... | auto | CPU or GPU |
| Primitives | `ops.expv`, `ops.expect`, `ops.eigs` | auto | CPU, GPU, or QPU |
| **GPU dialect** | `from uniqx.dialects import gpu` | gpu | GPU (cuBLAS / cuSOLVER) |
| **TPU dialect** | `from uniqx.dialects import tpu` | tpu | TPU |
| **Quantum dialect** | `from uniqx.dialects import quantum` | sim/qpu | qsim or cuQuantum |

### Key Patterns

1. **One `@tracing.to_module` function, multiple dialects** — the tracer doesn't care which dialect an op comes from.
2. **Classical → quantum**: encode classical values as rotation angles (`quantum.ry(theta, q)`).
3. **Quantum → classical**: `quantum.measure(q)` returns a classical bit that can feed into `ops.*`.
4. **Primitives bridge both worlds**: `ops.expv(H, psi, t)` takes classical matrices and returns classical vectors, but may execute on QPU internally.
5. **uniqx decides hardware**: each node in the DAG gets assigned independently based on the cost model.